# Ders 5A — En İyi Adayların Toplanması

Bu notebook, önceki adımların sonuç CSV dosyalarını `molfest_outputs/` klasöründen toplar.

Bu sürümde sonuç dosyaları iki yerden okunabilir:

1. Local çalışma klasörü: `molfest_outputs/...`
2. GitHub raw kaynak: `https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/...`

Bu nedenle Colab içinde `molfest_outputs/` klasörünü ayrıca indirmeden de çalışabilir.

Bu güncel sürümde progression başlangıcı `02_RF_Morgan_Baseline` olarak ayarlanmıştır. Yani iki veri seti için de ilk referans/gatekeeper sonuç `RandomForest + Morgan fingerprint` sonucudur. Sonraki adımlarda bu referansın üzerine feature ablation, feature selection, resampling, tuning ve advanced ensemble sonuçları eklenir.

## 1. Paket kontrolü

Bu scriptte yalnızca sonuç dosyalarını okumak, tablo düzenlemek ve grafik çizmek için gerekli paketler kullanılır.

In [ ]:
import sys
# Mevcut Python ortamında pip komutunu çalıştırmak için kullanılır.
import subprocess
# Eksik paketleri notebook içinden kurmak için kullanılır.
import importlib.util
# Bir paketin kurulu olup olmadığını kontrol etmek için kullanılır.

def install_if_missing(import_name, pip_name=None):
    """Eksik paket varsa kurar; paket zaten varsa işlem yapmaz."""
    pip_name = pip_name or import_name
    # pip adı verilmezse import adı paket adı olarak kullanılır.
    if importlib.util.find_spec(import_name) is None:
        # Paket kurulu değilse pip kurulumu yapılır.
        print(f"[INSTALL] {pip_name}")
        # Hangi paketin kurulacağı ekrana yazdırılır.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Paket sessiz modda kurulur.
    else:
        print(f"[OK] {import_name}")
        # Paket zaten kuruluysa tekrar kurulmaz.

required_packages = [
    ("pandas", "pandas"),
    # CSV okuma ve tablo işlemleri için kullanılır.
    ("numpy", "numpy"),
    # Delta/gain hesapları için kullanılır.
    ("matplotlib", "matplotlib"),
    # Progression grafiklerini çizmek için kullanılır.
]
# Bu scriptte gerçekten kullanılan paketler listelenir.

for import_name, pip_name in required_packages:
    # Paketler tek tek kontrol edilir.
    install_if_missing(import_name, pip_name)
    # Eksik paket varsa kurulur.

print("Paket kontrolü tamamlandı.")
# Paket kontrolünün bittiği yazdırılır.

## 2. Importlar ve genel ayarlar

Burada `molfest_outputs/` için hem local hem GitHub raw kaynak tanımlanır.

Önce local dosya aranır. Local dosya yoksa aynı dosya GitHub raw üzerinden okunur.

In [ ]:
from pathlib import Path
# Dosya ve klasör yollarını güvenli şekilde yönetmek için kullanılır.
import warnings
# Gereksiz uyarıları kontrol etmek için kullanılır.
warnings.filterwarnings("ignore")
# Notebook çıktısını sade tutmak için uyarılar gizlenir.

import numpy as np
# Sayısal hesaplamalar için kullanılır.
import pandas as pd
# CSV okuma, tablo düzenleme ve sonuç kaydetme için kullanılır.
import matplotlib.pyplot as plt
# ROC progression grafiklerini çizmek için kullanılır.

DATASETS = ["ERa_BLA_assay", "ERa_LUC_VM7_assay"]
# Raporlanacak iki pipeline/veri seti tanımlanır.

OUTPUT_ROOT = Path("molfest_outputs")
# Localde sonuçların aranacağı ana klasör.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# 09 çıktıları için ana klasör yoksa oluşturulur.

LESSON_OUT = OUTPUT_ROOT / "09_collect_best_candidates"
# Bu notebookun sonuç klasörü belirlenir.
LESSON_OUT.mkdir(parents=True, exist_ok=True)
# Sonuç klasörü yoksa oluşturulur.

GITHUB_OUTPUTS_BASE_URL = "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs"
# GitHub'a yüklenen tek output klasörünün raw URL kökü.

STEP_ORDER = {
    "02_rf_morgan_baseline": 2,
    "03_feature_ablation": 3,
    "04_feature_selection": 4,
    "05_model_search": 5,
    "06_resampling": 6,
    "07_tuning": 7,
    "08_advanced_ensemble": 8,
}
# Adımların grafik ve tablo sırası tanımlanır. Progression artık 02 baseline ile başlar.

print("Ayarlar hazır.")
# Ayar hücresinin tamamlandığı yazdırılır.
print("GitHub output kaynağı:", GITHUB_OUTPUTS_BASE_URL)
# GitHub raw output kökü ekrana yazdırılır.

## 3. Yardımcı fonksiyonlar

Bu fonksiyonlar:

- Local dosya varsa localden okur.
- Localde yoksa GitHub raw linkinden okur.
- Farklı adımlardan gelen tabloları ortak aday formatına çevirir.
- `RandomForest + Morgan` referans satırını `02_rf_morgan_baseline` olarak ekler.

In [ ]:
def show_table(df, n=50, title=None):
    """Sonuç tablosunu Colab'da normal pandas tablo formatında gösterir."""
    if title:
        # Tablo başlığı varsa önce başlık yazdırılır.
        print("\n" + title)
    try:
        # Colab/Jupyter ortamında display fonksiyonu varsa normal tablo olarak gösterilir.
        display(df.head(n))
    except NameError:
        # Terminalde çalıştırılırsa tablo metin olarak yazdırılır.
        print(df.head(n).to_string(index=False))


def github_raw_url(relative_path):
    """molfest_outputs içindeki göreli yolu GitHub raw URL'ye çevirir."""
    relative_path = str(relative_path).replace("\\", "/").lstrip("/")
    # Yol ayraçları raw URL formatına uygun hale getirilir.
    if relative_path.startswith("molfest_outputs/"):
        # Kullanıcı yanlışlıkla molfest_outputs ile başlatırsa bu kısım çıkarılır.
        relative_path = relative_path[len("molfest_outputs/"):]
    return f"{GITHUB_OUTPUTS_BASE_URL}/{relative_path}"
    # Tam GitHub raw CSV linki döndürülür.


def read_csv_local_or_github(relative_path):
    """Önce local molfest_outputs içinde arar; yoksa GitHub raw üzerinden okur."""
    relative_path = str(relative_path).replace("\\", "/").lstrip("/")
    # Göreli yol string haline getirilir.
    if relative_path.startswith("molfest_outputs/"):
        # Yol zaten molfest_outputs ile başlıyorsa tekrar eklememek için temizlenir.
        relative_path = relative_path[len("molfest_outputs/"):]

    local_path = OUTPUT_ROOT / relative_path
    # Localde aranacak dosya yolu oluşturulur.

    if local_path.exists():
        # Dosya localde varsa localden okunur.
        print(f"[LOCAL] okundu: {local_path}")
        # Local okuma bilgisi yazdırılır.
        return pd.read_csv(local_path), str(local_path)
        # CSV tablosu ve kaynak bilgisi döndürülür.

    url = github_raw_url(relative_path)
    # Local dosya yoksa GitHub raw URL hazırlanır.
    try:
        # GitHub raw CSV okunmaya çalışılır.
        df = pd.read_csv(url)
        # CSV tablo olarak okunur.
        print(f"[GITHUB] okundu: {url}")
        # GitHub okuma bilgisi yazdırılır.
        return df, url
        # CSV tablosu ve kaynak bilgisi döndürülür.
    except Exception as e:
        # GitHub'da da bulunamazsa bu kaynak atlanır.
        print(f"[SKIP] bulunamadı: {relative_path}")
        # Hangi dosyanın bulunamadığı yazdırılır.
        print(f"       GitHub denendi: {url}")
        # Denenen raw URL yazdırılır.
        print(f"       Hata: {type(e).__name__}: {e}")
        # Kısa hata bilgisi yazdırılır.
        return None, url
        # Eksik kaynak için None döndürülür.


def read_first_available(relative_paths):
    """Alternatif dosya adlarından ilk okunabilen CSV dosyasını döndürür."""
    for relative_path in relative_paths:
        # Olası dosya yolları sırayla denenir.
        df, source = read_csv_local_or_github(relative_path)
        # Dosya local veya GitHub üzerinden okunmaya çalışılır.
        if df is not None and not df.empty:
            # Okunan tablo boş değilse bu kaynak kullanılır.
            return df, source
            # Tablo ve kaynak bilgisi döndürülür.
    return None, str(relative_paths[0])
    # Hiçbiri okunamazsa None döndürülür.


def get_value(row, names, default=""):
    """Birden fazla olası kolon adından ilk bulunan değeri alır."""
    for name in names:
        # Olası kolon adları sırayla denenir.
        if name in row.index:
            # Kolon varsa değeri döndürülür.
            return row[name]
    return default
    # Hiçbiri yoksa varsayılan değer döndürülür.


def normalize_rows(df, step_name, source_file):
    """Farklı adımlardan gelen tabloları ortak aday formatına çevirir."""
    rows = []
    # Standartlaştırılmış aday satırları burada tutulur.
    if df is None or df.empty:
        # Boş ya da eksik tablo varsa boş liste döndürülür.
        return rows

    for _, row in df.iterrows():
        # Kaynak tablodaki her satır aday olarak işlenir.
        dataset = get_value(row, ["Dataset"], "")
        # Veri seti adı alınır.
        if dataset not in DATASETS:
            # Bilinmeyen veri seti satırı varsa atlanır.
            continue

        if "ROC" not in row.index:
            # ROC yoksa performans adayı olarak kullanılamaz.
            continue

        comparison = get_value(row, ["Comparison"], "")
        # Comparison kolonu varsa alınır.
        model_name = get_value(row, ["ModelName", "ModelType"], comparison)
        # Model adı mümkün olan kolonlardan alınır.
        model_family = get_value(row, ["ModelFamily", "ModelType"], "")
        # Model ailesi mümkün olan kolonlardan alınır.
        feature_strategy = get_value(row, ["FeatureStrategy", "FeatureSet"], "")
        # Feature stratejisi mümkün olan kolonlardan alınır.
        sampling_scenario = get_value(row, ["SamplingScenario"], "none")
        # Sampling senaryosu alınır; yoksa none yazılır.

        if comparison:
            # Comparison varsa aday etiketi onun üzerinden başlar.
            candidate_label = comparison
        elif model_name:
            # Model adı varsa aday etiketi model adıdır.
            candidate_label = model_name
        else:
            # Son seçenek olarak feature stratejisi yazılır.
            candidate_label = feature_strategy

        if candidate_label in ["Best advanced ensemble", "Gatekeeper"] and model_name:
            # Advanced ensemble karşılaştırmalarında model adı etikete eklenir.
            candidate_label = f"{candidate_label}: {model_name}"

        rows.append({
            "Dataset": dataset,
            "Step": step_name,
            "StepOrder": STEP_ORDER[step_name],
            "CandidateLabel": candidate_label,
            "ModelName": model_name,
            "ModelFamily": model_family,
            "FeatureStrategy": feature_strategy,
            "SamplingScenario": sampling_scenario,
            "ROC": row.get("ROC", np.nan),
            "AP": row.get("AP", np.nan),
            "F1": row.get("F1", np.nan),
            "BalancedAccuracy": row.get("BalancedAccuracy", np.nan),
            "SourceFile": str(source_file),
        })
        # Standart aday satırı listeye eklenir.

    return rows
    # Standartlaştırılmış aday satırları döndürülür.


def make_step02_baseline_rows(all_candidates_df):
    """03 içindeki Morgan reference satırını 02 baseline satırı olarak kopyalar."""
    baseline_rows = []
    # İki veri seti için baseline satırları burada tutulur.

    for dataset_key in DATASETS:
        # Her veri seti için ayrı baseline aranır.
        dataset_candidates = all_candidates_df[all_candidates_df["Dataset"] == dataset_key].copy()
        # İlgili veri setinin adayları alınır.
        reference_candidates = dataset_candidates[
            (dataset_candidates["Step"] == "03_feature_ablation")
            & (
                dataset_candidates["CandidateLabel"].astype(str).str.contains("Reference", case=False, na=False)
                | dataset_candidates["FeatureStrategy"].astype(str).str.lower().eq("morgan")
            )
        ].copy()
        # 03 dosyasındaki RF + Morgan reference satırı aranır.

        if reference_candidates.empty:
            # Reference bulunamazsa bu veri seti için baseline eklenmez.
            print(f"[UYARI] {dataset_key} için RF + Morgan reference satırı bulunamadı.")
            continue

        baseline = reference_candidates.sort_values("ROC", ascending=False).iloc[0].copy()
        # Reference satırları içinde en yüksek ROC-AUC'ye sahip olan alınır.
        baseline["Step"] = "02_rf_morgan_baseline"
        # Progression başlangıcı olarak step 02 adı verilir.
        baseline["StepOrder"] = STEP_ORDER["02_rf_morgan_baseline"]
        # Step sırası 2 yapılır.
        baseline["CandidateLabel"] = "RF + Morgan baseline"
        # Öğretim akışına uygun aday etiketi verilir.
        baseline["ModelName"] = "RandomForest"
        # Baseline model adı netleştirilir.
        baseline["ModelFamily"] = "RandomForest"
        # Baseline model ailesi netleştirilir.
        baseline["FeatureStrategy"] = "morgan"
        # Baseline feature stratejisi netleştirilir.
        baseline["SamplingScenario"] = "none"
        # Baseline sampling kullanmaz.
        baseline_rows.append(baseline)
        # Baseline satırı listeye eklenir.

    return pd.DataFrame(baseline_rows)
    # Baseline satırları tablo olarak döndürülür.


def plot_progression(df, dataset_key, out_file):
    """Seçilen adayların ROC progression grafiğini çizer."""
    plot_df = df[df["Dataset"] == dataset_key].sort_values("StepOrder").copy()
    # İlgili veri setinin seçilmiş adım sonuçları alınır.
    if plot_df.empty:
        # Veri yoksa grafik çizilmez.
        return

    plt.figure(figsize=(9, 4))
    # Grafik boyutu belirlenir.
    plt.plot(plot_df["Step"], plot_df["ROC"], marker="o")
    # Adım adım ROC-AUC çizilir.
    plt.xticks(rotation=35, ha="right")
    # X ekseni etiketleri okunabilir olması için döndürülür.
    plt.ylim(bottom=0.8)
    # ROC farkları daha net görünsün diye grafik 0.8'ten başlatılır.
    plt.ylabel("ROC-AUC")
    # Y ekseni etiketi yazılır.
    plt.title(f"{dataset_key} — seçilen aday progression")
    # Grafik başlığı yazılır.
    plt.tight_layout()
    # Yerleşim düzenlenir.
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    # Grafik PNG olarak kaydedilir.
    plt.show()
    # Grafik ekranda gösterilir.

## 4. Önceki adım çıktılarının toplanması

Bu hücre 03–08 adımlarının CSV sonuçlarını toplar.

Dosya localde yoksa GitHub raw üzerinden okunur. Böylece `molfest_outputs/` GitHub'a yüklendikten sonra Colab'da ekstra indirme yapmadan çalışır.

Progression için başlangıç satırı `02_rf_morgan_baseline` olarak ayrıca oluşturulur.

In [ ]:
candidate_sources = [
    (
        "03_feature_ablation",
        [
            "03_rf_feature_ablation/03_morgan_reference_vs_best_feature_set_all.csv",
            "03_rf_feature_ablation/03_gain_vs_baseline.csv",
            "03_rf_feature_ablation/03_best_feature_set_per_dataset.csv",
        ],
    ),
    # Step 03: Morgan reference vs en iyi feature set.
    (
        "04_feature_selection",
        [
            "04_feature_selection/04_gatekeeper_vs_best_feature_selection_all.csv",
            "04_feature_selection/04_best_feature_selection_per_dataset.csv",
            "04_feature_selection/04_all_feature_selection_combinations.csv",
        ],
    ),
    # Step 04: Feature selection denemeleri.
    (
        "05_model_search",
        [
            "05_train_12_models/05_gatekeeper_vs_best_12_model_all.csv",
            "05_train_12_models/05_best_model_per_dataset.csv",
            "05_train_12_models/05_12_model_metrics_all.csv",
        ],
    ),
    # Step 05: 12 model araması ve RF ile devam kararı.
    (
        "06_resampling",
        [
            "06_resampling/06_gatekeeper_vs_best_rf_sampling_all.csv",
            "06_resampling/06_best_rf_sampling_per_dataset.csv",
            "06_resampling/06_rf_sampling_metrics_all.csv",
        ],
    ),
    # Step 06: RF sampling senaryoları.
    (
        "07_tuning",
        [
            "07_random_search_tuning_fast/07_sampling_gatekeeper_vs_tuned_rf_all.csv",
            "07_random_search_tuning_fast/07_gain_vs_sampling_gatekeeper.csv",
            "07_random_search_tuning_fast/07_rf_tuning_metrics.csv",
        ],
    ),
    # Step 07: RF random search tuning.
    (
        "08_advanced_ensemble",
        [
            "08_advanced_ensembles/08_gatekeeper_vs_best_advanced_ensemble.csv",
            "08_advanced_ensembles/08_all_advanced_ensemble_results.csv",
        ],
    ),
    # Step 08: Advanced voting/stacking/feature ensemble/multiview.
]
# Okunacak aday kaynakları adım sırasıyla tanımlanır.

all_candidate_rows = []
# Bütün aday satırları burada tutulur.
source_report_rows = []
# Hangi adımın hangi kaynaktan okunduğu burada tutulur.

for step_name, source_paths in candidate_sources:
    # Her adımın CSV sonucu sırayla okunur.
    df, source = read_first_available(source_paths)
    # Alternatif dosyalardan ilk okunabilen kaynak alınır.

    status = "OK" if df is not None and not df.empty else "MISSING"
    # Kaynak durum bilgisi belirlenir.
    source_report_rows.append({
        "Step": step_name,
        "Status": status,
        "SourceUsed": source,
        "Rows": 0 if df is None else len(df),
    })
    # Kaynak rapor satırı eklenir.

    all_candidate_rows.extend(normalize_rows(df, step_name, source))
    # Kaynak tablodaki satırlar ortak aday formatına çevrilip genel listeye eklenir.

source_report_df = pd.DataFrame(source_report_rows)
# Kaynak okuma raporu tabloya çevrilir.
show_table(source_report_df, n=20, title="Okunan sonuç kaynakları")
# Hangi dosyaların local/GitHub üzerinden okunduğu gösterilir.

all_candidates_df = pd.DataFrame(all_candidate_rows)
# Bütün adaylar tabloya çevrilir.

if all_candidates_df.empty:
    # Hiç aday bulunamazsa açık hata verilir.
    raise RuntimeError(
        "Hiç aday sonucu bulunamadı. GitHub'da molfest_outputs klasörünün ve CSV dosyalarının yüklü olduğundan emin olun."
    )

baseline_df = make_step02_baseline_rows(all_candidates_df)
# 03 içindeki RF + Morgan reference satırı 02 baseline olarak kopyalanır.

if not baseline_df.empty:
    # Baseline satırı oluşturulduysa bütün adaylara eklenir.
    all_candidates_df = pd.concat([baseline_df, all_candidates_df], ignore_index=True)
    # 02 baseline artık aday tablosunun parçasıdır.

all_candidates_df = all_candidates_df.sort_values(["Dataset", "ROC"], ascending=[True, False])
# Bütün adaylar veri seti içinde ROC-AUC en yüksek en üstte olacak şekilde sıralanır.

selected_by_step_df = (
    all_candidates_df
    .sort_values(["Dataset", "StepOrder", "ROC"], ascending=[True, True, False])
    .groupby(["Dataset", "Step"], as_index=False)
    .head(1)
    .sort_values(["Dataset", "StepOrder"])
)
# Her veri seti ve her adım için ROC-AUC temelli en iyi aday seçilir.
# Bu tablo progression için step sırasına göre kalır; yani figür 02 baseline'dan başlar.

final_candidate_df = (
    selected_by_step_df
    .sort_values(["Dataset", "StepOrder"])
    .groupby("Dataset", as_index=False)
    .tail(1)
    .sort_values(["Dataset", "ROC"], ascending=[True, False])
)
# Her veri setinin son adımda seçilen final adayı alınır.

source_report_df.to_csv(LESSON_OUT / "09_source_reading_report.csv", index=False)
# Hangi kaynakların okunduğu CSV olarak kaydedilir.
all_candidates_df.to_csv(LESSON_OUT / "09_all_candidate_steps.csv", index=False)
# Bütün adaylar CSV olarak kaydedilir.
selected_by_step_df.to_csv(LESSON_OUT / "09_selected_progression_by_step.csv", index=False)
# Her adımda seçilen adaylar CSV olarak kaydedilir.
final_candidate_df.to_csv(LESSON_OUT / "09_final_selected_candidate_per_dataset.csv", index=False)
# Final adaylar CSV olarak kaydedilir.

for dataset_key in DATASETS:
    # İki pipeline ayrı ayrı raporlanır.
    print("\n" + "#" * 100)
    # Pipeline ayıracı yazdırılır.
    print(f"PIPELINE: {dataset_key}")
    # Hangi veri seti/pipeline raporlandığı yazdırılır.
    print("#" * 100)
    # Pipeline ayıracı tamamlanır.

    dataset_candidates = all_candidates_df[all_candidates_df["Dataset"] == dataset_key].copy()
    # İlgili veri setinin bütün adayları alınır.
    dataset_candidates = dataset_candidates.sort_values("ROC", ascending=False)
    # Bütün aday tablosu ROC-AUC en yüksek en üstte olacak şekilde sıralanır.
    show_table(
        dataset_candidates[
            [
                "Dataset",
                "Step",
                "CandidateLabel",
                "ModelName",
                "ModelFamily",
                "FeatureStrategy",
                "SamplingScenario",
                "ROC",
                "AP",
                "F1",
                "SourceFile",
            ]
        ],
        n=100,
        title=f"{dataset_key} — bütün adaylar ROC-AUC sıralı"
    )
    # Veri setinin bütün adayları ayrı tablo olarak, ROC-AUC yüksekten düşüğe gösterilir.

    dataset_selected = selected_by_step_df[selected_by_step_df["Dataset"] == dataset_key].copy()
    # İlgili veri setinin adım adım seçilen adayları alınır.
    show_table(
        dataset_selected[
            [
                "Dataset",
                "Step",
                "CandidateLabel",
                "ModelName",
                "ModelFamily",
                "FeatureStrategy",
                "SamplingScenario",
                "ROC",
                "AP",
                "F1",
            ]
        ],
        title=f"{dataset_key} — adım adım seçilen adaylar"
    )
    # Adım adım seçilen adaylar step sırasına göre gösterilir.

    plot_progression(
        selected_by_step_df,
        dataset_key,
        LESSON_OUT / f"{dataset_key}_selected_progression_roc.png",
    )
    # Seçilen adayların ROC progression grafiği çizilir.

show_table(
    final_candidate_df[
        [
            "Dataset",
            "Step",
            "CandidateLabel",
            "ModelName",
            "ModelFamily",
            "FeatureStrategy",
            "SamplingScenario",
            "ROC",
            "AP",
            "F1",
        ]
    ].sort_values("ROC", ascending=False),
    title="09 — final seçilen adaylar ROC-AUC sıralı"
)
# İki veri seti için final seçilen adaylar ROC-AUC yüksekten düşüğe gösterilir.